# Optuna baselines — TPE and GPSampler on the Italian retail task

Modern Bayesian-optimization baselines, run on THIS machine, replacing the
prototype's skopt `gp_minimize` (archived; cannot run on numpy 2.x). Same data
pipeline, same discrete grid (n_estimators 10..260 step 10 — prototype Run A's 26-value grid), same `TimeSeriesSplit(5)`, same MAE scoring, same
sklearn 1.9 environment as the prototype and PatternSearchCV runs.

Budget: 15 trials per sampler — matching the recorded `gp_minimize(n_calls=15)`
budget. Every Optuna trial evaluates on 100% of the rows (no multi-fidelity),
so full-fit equivalents = trials. `best after 11 trials` is also reported to
compare at the prototype's evaluation budget.

In [1]:
# --- Data pipeline: identical to the prototype notebook (path localized) ---
import time
import numpy as np
import pandas as pd

trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")

SplitPoint = int(len(trainBench.index) * 0.8)
df = trainBench
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)
for col in ["Max_Gust_SpeedKm_h", "CloudCover", "Max_VisibilityKm",
            "Min_VisibilitykM", "Mean_VisibilityKm"]:
    df = df.drop(col, axis=1)
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
del df
mask = trainBench.columns.difference(["NumberOfSales"])
X = trainBench[mask]
y = trainBench["NumberOfSales"]
del trainBench, validBench
print("X:", X.shape)


C:\Users\Home\AppData\Local\Temp\ipykernel_18700\2456675003.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


X: (418416, 30)


In [2]:
# --- Objective: same estimator, grid, CV and scoring as every other run ---
import optuna
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        # identical discrete grid to the prototype / PatternSearchCV runs
        "n_estimators": trial.suggest_int("n_estimators", 10, 260, step=10),  # Run A grid: 26 values
        "max_features": trial.suggest_int("max_features", 2, 4),
        "max_depth": trial.suggest_int("max_depth", 5, 17),
    }
    clf = ExtraTreesRegressor(n_jobs=1, random_state=0, **params)
    scores = cross_val_score(clf, X, y, cv=tscv,
                             scoring="neg_mean_absolute_error", n_jobs=-1)
    return -scores.mean()  # MAE, minimized

def run_study(sampler, name, n_trials=15):
    study = optuna.create_study(direction="minimize", sampler=sampler,
                                study_name=name)
    t0 = time.time()
    study.optimize(objective, n_trials=n_trials)
    elapsed = time.time() - t0
    vals = [t.value for t in study.trials]
    best_at_11 = min(vals[:11])
    print(f"{name}: best MAE {study.best_value:.3f} at {study.best_params}")
    print(f"{name}: best after 11 trials (prototype budget): {best_at_11:.3f}")
    print(f"{name}: wall-clock {elapsed:.1f} s for {n_trials} trials")
    return {"name": name, "elapsed": elapsed, "best": study.best_value,
            "best_params": study.best_params, "best_at_11": best_at_11,
            "values": vals}

optuna.logging.set_verbosity(optuna.logging.WARNING)


C:\FILES\Code\Benchmarking\Working_on_Train_Set\V2025\pattern-search-cv\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
results = []
import json, pathlib
def save_partial():
    pathlib.Path("optuna_partial_results.json").write_text(json.dumps(results, default=float))
try:
    results.append(run_study(optuna.samplers.TPESampler(seed=0), "TPE", 15))
    save_partial()
except Exception as e:
    print(f"TPE FAILED: {type(e).__name__}: {e}")

TPE: best MAE 810.553 at {'n_estimators': 100, 'max_features': 4, 'max_depth': 17}
TPE: best after 11 trials (prototype budget): 811.500
TPE: wall-clock 828.7 s for 15 trials


In [4]:
try:
    results.append(run_study(optuna.samplers.GPSampler(seed=0, deterministic_objective=True), "GP", 15))
    save_partial()
except Exception as e:
    print(f"GP SKIPPED: {type(e).__name__}: {e}")
    print("(GPSampler needs torch; will be run separately from a short-path venv)")

C:\Users\Home\AppData\Local\Temp\ipykernel_18700\3666594468.py:2: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  results.append(run_study(optuna.samplers.GPSampler(seed=0, deterministic_objective=True), "GP", 15))


[W 2026-07-14 09:43:44,000] Trial 10 failed with parameters: {} because of the following error: ModuleNotFoundError("No module named 'torch'").
Traceback (most recent call last):
  File "C:\FILES\Code\Benchmarking\Working_on_Train_Set\V2025\pattern-search-cv\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Home\AppData\Local\Temp\ipykernel_18700\2889872221.py", line 11, in objective
    "n_estimators": trial.suggest_int("n_estimators", 10, 260, step=10),  # Run A grid: 26 values
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\FILES\Code\Benchmarking\Working_on_Train_Set\V2025\pattern-search-cv\.venv\Lib\site-packages\optuna\_convert_positional_args.py", line 127, in converter_wrapper
    return func(**kwargs)  # type: ignore[call-arg]
           ^^^^^^^^^^^^^^
  File "C:\FILES\Code\Benchmarking\Working_on_Train_Set\V2025\pattern-search-cv\.v

[W 2026-07-14 09:43:44,026] Trial 10 failed with value None.


GP SKIPPED: ModuleNotFoundError: No module named 'torch'
(GPSampler needs torch; will be run separately from a short-path venv)


In [5]:
print("=" * 72)
print("OPTUNA BASELINES (15 trials each, every trial on 100% of rows)")
print("=" * 72)
print(f"{'method':26s}{'evals':>6s}{'full-fit eq':>12s}{'best MAE':>10s}{'wall-clock':>12s}")
for r in results:
    print(f"{'Optuna ' + r['name']:26s}{15:>6d}{15.0:>12.2f}{r['best']:>10.3f}"
          f"{r['elapsed']:>11.1f}s")
print(f"{'prototype pattern search':26s}{11:>6d}{11.0:>12.2f}{805.038:>10.3f}"
      f"{'(pending)':>12s}")
print(f"{'NEW PatternSearchCV':26s}{23:>6d}{6.80:>12.2f}{805.730:>10.3f}"
      f"{746.8:>11.1f}s")
print()
for r in results:
    print(f"Optuna {r['name']} trial-by-trial MAE: "
          + ", ".join(f"{v:.1f}" for v in r["values"]))

OPTUNA BASELINES (15 trials each, every trial on 100% of rows)
method                     evals full-fit eq  best MAE  wall-clock
Optuna TPE                    15       15.00   810.553      828.7s
prototype pattern search      11       11.00   805.038   (pending)
NEW PatternSearchCV           23        6.80   805.730      746.8s

Optuna TPE trial-by-trial MAE: 965.1, 1046.0, 811.5, 1020.0, 1325.3, 1112.9, 814.7, 975.0, 1415.8, 1168.3, 1408.8, 814.6, 810.6, 871.3, 815.7
